### Allscripts Sunrise (SCM) Procedure Occurrence Hydration

This notebook currently derives procedures from Sunrise orders/tasks so `procedure_occurrence` is no longer a commented-out template.

Client follow-up also provided a separate SCM billing extract pattern in `(Clone) procedures_SCM.py` that sources CPT/HCPCS-style procedures from Soarian, DSS, and Athena `omny_accounts` feeds through an SCM encounter mapper. Treat that billing flow as complementary source context that still needs reconciliation with this OMOP hydration path and its concept-mapping strategy.

In [0]:
# %sql
# TRUNCATE TABLE _exponent.omop_scm.procedure_occurrence

In [0]:
%sql
-- TRUNCATE TABLE _exponent.omop_scm.procedure_occurrence;


In [0]:
%sql
-- DELETE FROM _exponent.omop_silver.procedure_occurrence
-- WHERE source_system = 'allscripts_scm';


In [0]:
%sql
-- DELETE FROM _exponent.omop_mapping.source_to_procedure_occurrence
-- WHERE source_system = 'allscripts_scm';


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW silver_procedure_occurrence AS
WITH concept_relationship_maps_to_procedure_dedup AS (
  SELECT
    cr.concept_id_1 AS source_concept_id,
    cr.concept_id_2 AS standard_concept_id,
    ROW_NUMBER() OVER (
      PARTITION BY cr.concept_id_1
      ORDER BY cr.concept_id_2 ASC
    ) AS rn
  FROM _exponent.omop.concept_relationship cr
  INNER JOIN _exponent.omop.concept target_concept
    ON target_concept.concept_id = cr.concept_id_2
   AND target_concept.standard_concept = 'S'
   AND target_concept.invalid_reason IS NULL
   AND target_concept.domain_id = 'Procedure'
  WHERE cr.relationship_id = 'Maps to'
    AND cr.invalid_reason IS NULL
), procedure_mapping AS (
  SELECT
    source_concept.concept_id AS source_concept_id,
    UPPER(source_concept.concept_code) AS source_concept_code,
    COALESCE(
      mapped.standard_concept_id,
      CASE
        WHEN source_concept.standard_concept = 'S' THEN source_concept.concept_id
      END
    ) AS standard_concept_id
  FROM _exponent.omop.concept source_concept
  LEFT JOIN concept_relationship_maps_to_procedure_dedup mapped
    ON mapped.source_concept_id = source_concept.concept_id
   AND mapped.rn = 1
  WHERE source_concept.vocabulary_id IN ('CPT4', 'HCPCS')
    AND source_concept.domain_id = 'Procedure'
    AND source_concept.invalid_reason IS NULL
), procedure_code_candidates AS (
  SELECT order_guid, source_concept_code, source_priority
  FROM (
    SELECT
      ord.GUID AS order_guid,
      NULLIF(REGEXP_EXTRACT(UPPER(CAST(ord.IDCode AS STRING)), '([A-Z][0-9]{4}|[0-9]{4}[A-Z]|[0-9]{5})', 1), '') AS source_concept_code,
      1 AS source_priority
    FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
    UNION ALL
    SELECT
      ord.GUID AS order_guid,
      NULLIF(REGEXP_REPLACE(UPPER(CAST(ord.IDCode AS STRING)), '[^A-Z0-9]', ''), '') AS source_concept_code,
      2 AS source_priority
    FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
    UNION ALL
    SELECT
      ord.GUID AS order_guid,
      NULLIF(REGEXP_EXTRACT(UPPER(CAST(ord.Name AS STRING)), '([A-Z][0-9]{4}|[0-9]{4}[A-Z]|[0-9]{5})', 1), '') AS source_concept_code,
      3 AS source_priority
    FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
    UNION ALL
    SELECT
      oto.OrderGUID AS order_guid,
      NULLIF(REGEXP_EXTRACT(UPPER(CAST(oto.TaskName AS STRING)), '([A-Z][0-9]{4}|[0-9]{4}[A-Z]|[0-9]{5})', 1), '') AS source_concept_code,
      4 AS source_priority
    FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3ordertaskoccurrence oto
    WHERE oto.Active = TRUE
  )
  WHERE source_concept_code IS NOT NULL
), procedure_mapping_by_order AS (
  SELECT order_guid, source_concept_id, standard_concept_id
  FROM (
    SELECT
      candidate.order_guid,
      procedure_mapping.source_concept_id,
      procedure_mapping.standard_concept_id,
      ROW_NUMBER() OVER (
        PARTITION BY candidate.order_guid
        ORDER BY candidate.source_priority, procedure_mapping.standard_concept_id, procedure_mapping.source_concept_id
      ) AS rn
    FROM procedure_code_candidates candidate
    INNER JOIN procedure_mapping
      ON procedure_mapping.source_concept_code = candidate.source_concept_code
  )
  WHERE rn = 1
), billing_procedure_raw AS (
  SELECT
    'dss' AS billing_source,
    CAST(Patient_EPI AS STRING) AS patient_epi,
    CAST(Encounter_ID AS STRING) AS encounter_id,
    REGEXP_REPLACE(UPPER(CAST(CPT_Code AS STRING)), '[^A-Z0-9]', '') AS cpt_code,
    NULLIF(TRIM(CAST(Modifiers AS STRING)), '') AS modifier_source_value,
    TRY_CAST(service_date AS TIMESTAMP) AS service_datetime
  FROM _exponent._bronze_billing_dss.omny_accounts
  WHERE CPT_Code IS NOT NULL

  UNION ALL

  SELECT
    'soarian' AS billing_source,
    CAST(Patient_EPI AS STRING) AS patient_epi,
    CAST(Encounter_ID AS STRING) AS encounter_id,
    REGEXP_REPLACE(UPPER(CAST(CPT_Code AS STRING)), '[^A-Z0-9]', '') AS cpt_code,
    NULLIF(TRIM(CAST(Modifiers AS STRING)), '') AS modifier_source_value,
    TRY_CAST(service_date AS TIMESTAMP) AS service_datetime
  FROM _exponent._bronze_billing_soarian.omny_accounts
  WHERE CPT_Code IS NOT NULL
), billing_procedure_distinct AS (
  SELECT DISTINCT billing_source, patient_epi, encounter_id, cpt_code, modifier_source_value, service_datetime
  FROM billing_procedure_raw
  WHERE cpt_code RLIKE '^([0-9]{5}|[A-Z][0-9]{4})$'
    AND patient_epi IS NOT NULL
    AND service_datetime IS NOT NULL
), srgry_patient_identifier AS (
  SELECT DISTINCT
    TRIM(IDN_VAL_TXT) AS patient_epi,
    ptnt_dim_id
  FROM _exponent._bronze_srgry_dmart.rpt_ptnt_idn_arr_vw
  WHERE IDN_VAL_TXT IS NOT NULL
    AND ptnt_dim_id IS NOT NULL
), srgry_case_visit AS (
  SELECT DISTINCT
    c.ptnt_dim_id,
    c.case_fct_id,
    c.case_strt_ts,
    COALESCE(c.case_end_ts, c.case_strt_ts) AS case_end_ts,
    CAST(cv.IDN_VAL_NUM AS BIGINT) AS client_visit_guid
  FROM _exponent._bronze_srgry_dmart.rpt_case_vw c
  JOIN _exponent._bronze_srgry_dmart.rpt_case_vst_idn_arr_vw cv
    ON cv.case_fct_id = c.case_fct_id
  WHERE LOWER(cv.IDN_TP_DESC) = 'clientvisitguid'
    AND cv.IDN_VAL_NUM IS NOT NULL
    AND c.case_strt_ts IS NOT NULL
), billing_procedure_ranked AS (
  SELECT
    b.*,
    procedure_mapping.source_concept_id,
    procedure_mapping.standard_concept_id,
    scv.client_visit_guid,
    stvo.visit_occurrence_id,
    vo.person_id,
    ROW_NUMBER() OVER (
      PARTITION BY b.billing_source, b.patient_epi, b.encounter_id, b.cpt_code, COALESCE(b.modifier_source_value, ''), b.service_datetime
      ORDER BY ABS(DATEDIFF(CAST(b.service_datetime AS DATE), CAST(scv.case_strt_ts AS DATE))), scv.case_fct_id
    ) AS rn
  FROM billing_procedure_distinct b
  INNER JOIN procedure_mapping
    ON procedure_mapping.source_concept_code = b.cpt_code
  INNER JOIN srgry_patient_identifier spi
    ON spi.patient_epi = TRIM(b.patient_epi)
  INNER JOIN srgry_case_visit scv
    ON scv.ptnt_dim_id = spi.ptnt_dim_id
   AND CAST(b.service_datetime AS DATE) BETWEEN DATE_SUB(CAST(scv.case_strt_ts AS DATE), 1)
                                            AND DATE_ADD(CAST(scv.case_end_ts AS DATE), 1)
  INNER JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
    ON stvo.visit_occurrence_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'dbo_cv3clientvisit', 'GUID', CAST(scv.client_visit_guid AS STRING))
   AND stvo.source_system = 'allscripts_scm'
   AND stvo.active_flag = TRUE
  INNER JOIN _exponent.omop_scm.visit_occurrence vo
    ON vo.visit_occurrence_id = stvo.visit_occurrence_id
), order_staged AS (
  SELECT
    CONCAT_WS(CHR(31), 'allscripts_scm', 'dbo_cv3order', 'GUID', CAST(ord.GUID AS STRING)) AS procedure_occurrence_source_value,
    stp.person_id,
    COALESCE(procedure_mapping_by_order.standard_concept_id, proc_concept.omop_concept_id, 0) AS procedure_concept_id,
    CAST(COALESCE(oto.PerformedFromDtm, ord.PerformedDtm, ord.SignificantDtm, ord.RequestedDtm, ord.Entered, ord.CreatedWhen) AS DATE) AS procedure_date,
    COALESCE(oto.PerformedFromDtm, ord.PerformedDtm, ord.SignificantDtm, ord.RequestedDtm, ord.Entered, ord.CreatedWhen) AS procedure_datetime,
    32817 AS procedure_type_concept_id,
    COALESCE(mod_concept.omop_concept_id, 0) AS modifier_concept_id,
    CAST(1 AS DOUBLE) AS quantity,
    stpr.provider_id AS provider_id,
    COALESCE(stvo.visit_occurrence_id, vo.visit_occurrence_id) AS visit_occurrence_id,
    NULL AS visit_detail_id,
    COALESCE(NULLIF(TRIM(ord.IDCode), ''), NULLIF(TRIM(ord.Name), ''), NULLIF(TRIM(oto.TaskName), ''), ord.TypeCode) AS procedure_source_value,
    COALESCE(procedure_mapping_by_order.source_concept_id, 0) AS procedure_source_concept_id,
    NULLIF(TRIM(ord.Modifier), '') AS modifier_source_value,
    'allscripts_scm' AS source_system,
    CURRENT_TIMESTAMP() AS last_mod_tsp
  FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
  LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3ordertaskoccurrence oto
    ON oto.OrderGUID = ord.GUID
   AND oto.Active = TRUE
  INNER JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(COALESCE(oto.ClientGUID, ord.ClientGUID) AS STRING))
   AND stp.active_flag = TRUE
  LEFT JOIN _exponent.omop_mapping.source_to_provider stpr
    ON stpr.provider_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'dbo_cv3careprovider', 'GUID', CAST(COALESCE(oto.PerformedProviderGUID, oto.EnteredProviderGUID, ord.CareProviderGUID) AS STRING))
   AND stpr.active_flag = TRUE
  LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
    ON stvo.visit_occurrence_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'dbo_cv3clientvisit', 'GUID', CAST(ord.ClientVisitGUID AS STRING))
   AND stvo.source_system = 'allscripts_scm'
   AND stvo.active_flag = TRUE
  LEFT JOIN _exponent.omop_scm.visit_occurrence vo
    ON vo.visit_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'dbo_cv3clientvisit', 'GUID', CAST(ord.ClientVisitGUID AS STRING))
  LEFT JOIN procedure_mapping_by_order
    ON procedure_mapping_by_order.order_guid = ord.GUID
  LEFT JOIN _exponent.omop_mapping.domain_source_to_concept proc_concept
    ON proc_concept.domain_id = 'Procedure'
   AND proc_concept.source_system = 'allscripts_scm'
   AND proc_concept.source_id = COALESCE(NULLIF(TRIM(ord.IDCode), ''), NULLIF(TRIM(ord.Name), ''), NULLIF(TRIM(oto.TaskName), ''), ord.TypeCode)
  LEFT JOIN _exponent.omop_mapping.domain_source_to_concept mod_concept
    ON mod_concept.domain_id = 'Modifier'
   AND mod_concept.source_system = 'allscripts_scm'
   AND mod_concept.source_id = NULLIF(TRIM(ord.Modifier), '')
  WHERE ord.Active = TRUE
    AND ord.GUID IS NOT NULL
    AND COALESCE(oto.ClientGUID, ord.ClientGUID) IS NOT NULL
    AND COALESCE(oto.PerformedFromDtm, ord.PerformedDtm, ord.SignificantDtm, ord.RequestedDtm, ord.Entered, ord.CreatedWhen) IS NOT NULL
    AND UPPER(COALESCE(ord.OrderStatusCode, oto.TaskStatusCode, '')) NOT IN ('CAN', 'CANCELLED', 'CANCELED')
    AND UPPER(COALESCE(ord.TypeCode, '')) NOT IN ('MEDICATION', 'MED', 'PHARMACY', 'LAB', 'LABORATORY')
    AND (
      UPPER(COALESCE(ord.TypeCode, '')) NOT IN ('OTHER', 'DIAGNOSTIC')
      OR procedure_mapping_by_order.standard_concept_id IS NOT NULL
      OR COALESCE(proc_concept.omop_concept_id, 0) <> 0
    )
), billing_staged AS (
  SELECT
    CONCAT_WS(
      CHR(31),
      'allscripts_scm',
      CONCAT('billing_', billing_source),
      'patient_epi', patient_epi,
      'encounter_id', COALESCE(encounter_id, ''),
      'service_datetime', CAST(service_datetime AS STRING),
      'cpt', cpt_code,
      'modifier', COALESCE(modifier_source_value, ''),
      'visit_guid', CAST(client_visit_guid AS STRING)
    ) AS procedure_occurrence_source_value,
    person_id,
    standard_concept_id AS procedure_concept_id,
    CAST(service_datetime AS DATE) AS procedure_date,
    service_datetime AS procedure_datetime,
    32810 AS procedure_type_concept_id,
    0 AS modifier_concept_id,
    CAST(1 AS DOUBLE) AS quantity,
    NULL AS provider_id,
    visit_occurrence_id,
    NULL AS visit_detail_id,
    cpt_code AS procedure_source_value,
    source_concept_id AS procedure_source_concept_id,
    modifier_source_value,
    'allscripts_scm' AS source_system,
    CURRENT_TIMESTAMP() AS last_mod_tsp
  FROM billing_procedure_ranked
  WHERE rn = 1
), staged AS (
  SELECT * FROM order_staged
  UNION ALL
  SELECT * FROM billing_staged
), deduped AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY procedure_occurrence_source_value ORDER BY procedure_datetime DESC) AS rn
  FROM staged
)
SELECT
  procedure_occurrence_source_value,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value,
  source_system,
  last_mod_tsp
FROM deduped
WHERE rn = 1;

In [0]:
%sql
INSERT INTO _exponent.omop_silver.procedure_occurrence (
  procedure_occurrence_source_value,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value,
  source_system,
  last_mod_tsp
)
SELECT
  procedure_occurrence_source_value,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value,
  source_system,
  last_mod_tsp
FROM silver_procedure_occurrence;


In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_procedure_occurrence (
  source_system,
  procedure_occurrence_source_value,
  active_flag,
  created_tsp,
  last_mod_tsp
)
SELECT
  s.source_system,
  s.procedure_occurrence_source_value,
  TRUE,
  CURRENT_TIMESTAMP(),
  COALESCE(s.last_mod_tsp, CURRENT_TIMESTAMP())
FROM silver_procedure_occurrence s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_procedure_occurrence x
  ON s.procedure_occurrence_source_value = x.procedure_occurrence_source_value
 AND x.source_system = 'allscripts_scm';


In [0]:
%sql
MERGE INTO _exponent.omop_scm.procedure_occurrence AS target
USING (
  SELECT
    spo.procedure_occurrence_id,
    s.person_id,
    s.procedure_concept_id,
    s.procedure_date,
    s.procedure_datetime,
    s.procedure_type_concept_id,
    s.modifier_concept_id,
    s.quantity,
    s.provider_id,
    s.visit_occurrence_id,
    s.visit_detail_id,
    s.procedure_source_value,
    s.procedure_source_concept_id,
    s.modifier_source_value
  FROM silver_procedure_occurrence s
  JOIN _exponent.omop_mapping.source_to_procedure_occurrence spo
    ON spo.procedure_occurrence_source_value = s.procedure_occurrence_source_value
   AND spo.source_system = 'allscripts_scm'
   AND spo.active_flag = TRUE
) AS source
ON target.procedure_occurrence_id = source.procedure_occurrence_id

WHEN MATCHED AND (
     NOT (target.person_id <=> source.person_id)
  OR NOT (target.procedure_concept_id <=> source.procedure_concept_id)
  OR NOT (target.procedure_date <=> source.procedure_date)
  OR NOT (target.procedure_datetime <=> source.procedure_datetime)
  OR NOT (target.procedure_type_concept_id <=> source.procedure_type_concept_id)
  OR NOT (target.modifier_concept_id <=> source.modifier_concept_id)
  OR NOT (target.quantity <=> source.quantity)
  OR NOT (target.provider_id <=> source.provider_id)
  OR NOT (target.visit_occurrence_id <=> source.visit_occurrence_id)
  OR NOT (target.visit_detail_id <=> source.visit_detail_id)
  OR NOT (target.procedure_source_value <=> source.procedure_source_value)
  OR NOT (target.procedure_source_concept_id <=> source.procedure_source_concept_id)
  OR NOT (target.modifier_source_value <=> source.modifier_source_value)
)
THEN UPDATE SET
  target.person_id = source.person_id,
  target.procedure_concept_id = source.procedure_concept_id,
  target.procedure_date = source.procedure_date,
  target.procedure_datetime = source.procedure_datetime,
  target.procedure_type_concept_id = source.procedure_type_concept_id,
  target.modifier_concept_id = source.modifier_concept_id,
  target.quantity = source.quantity,
  target.provider_id = source.provider_id,
  target.visit_occurrence_id = source.visit_occurrence_id,
  target.visit_detail_id = source.visit_detail_id,
  target.procedure_source_value = source.procedure_source_value,
  target.procedure_source_concept_id = source.procedure_source_concept_id,
  target.modifier_source_value = source.modifier_source_value

WHEN NOT MATCHED THEN INSERT (
  procedure_occurrence_id,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value
)
VALUES (
  source.procedure_occurrence_id,
  source.person_id,
  source.procedure_concept_id,
  source.procedure_date,
  source.procedure_datetime,
  source.procedure_type_concept_id,
  source.modifier_concept_id,
  source.quantity,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.procedure_source_value,
  source.procedure_source_concept_id,
  source.modifier_source_value
);

In [0]:
%sql
MERGE INTO _exponent.omop_allscripts.procedure_occurrence AS target
USING (
  SELECT
    spo.procedure_occurrence_id,
    s.person_id,
    s.procedure_concept_id,
    s.procedure_date,
    s.procedure_datetime,
    s.procedure_type_concept_id,
    s.modifier_concept_id,
    s.quantity,
    s.provider_id,
    s.visit_occurrence_id,
    s.visit_detail_id,
    s.procedure_source_value,
    s.procedure_source_concept_id,
    s.modifier_source_value
  FROM silver_procedure_occurrence s
  JOIN _exponent.omop_mapping.source_to_procedure_occurrence spo
    ON spo.procedure_occurrence_source_value = s.procedure_occurrence_source_value
   AND spo.source_system = 'allscripts_scm'
   AND spo.active_flag = TRUE
) AS source
ON target.procedure_occurrence_id = source.procedure_occurrence_id

WHEN MATCHED AND (
     NOT (target.person_id <=> source.person_id)
  OR NOT (target.procedure_concept_id <=> source.procedure_concept_id)
  OR NOT (target.procedure_date <=> source.procedure_date)
  OR NOT (target.procedure_datetime <=> source.procedure_datetime)
  OR NOT (target.procedure_type_concept_id <=> source.procedure_type_concept_id)
  OR NOT (target.modifier_concept_id <=> source.modifier_concept_id)
  OR NOT (target.quantity <=> source.quantity)
  OR NOT (target.provider_id <=> source.provider_id)
  OR NOT (target.visit_occurrence_id <=> source.visit_occurrence_id)
  OR NOT (target.visit_detail_id <=> source.visit_detail_id)
  OR NOT (target.procedure_source_value <=> source.procedure_source_value)
  OR NOT (target.procedure_source_concept_id <=> source.procedure_source_concept_id)
  OR NOT (target.modifier_source_value <=> source.modifier_source_value)
)
THEN UPDATE SET
  target.person_id = source.person_id,
  target.procedure_concept_id = source.procedure_concept_id,
  target.procedure_date = source.procedure_date,
  target.procedure_datetime = source.procedure_datetime,
  target.procedure_type_concept_id = source.procedure_type_concept_id,
  target.modifier_concept_id = source.modifier_concept_id,
  target.quantity = source.quantity,
  target.provider_id = source.provider_id,
  target.visit_occurrence_id = source.visit_occurrence_id,
  target.visit_detail_id = source.visit_detail_id,
  target.procedure_source_value = source.procedure_source_value,
  target.procedure_source_concept_id = source.procedure_source_concept_id,
  target.modifier_source_value = source.modifier_source_value

WHEN NOT MATCHED THEN INSERT (
  procedure_occurrence_id,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value
)
VALUES (
  source.procedure_occurrence_id,
  source.person_id,
  source.procedure_concept_id,
  source.procedure_date,
  source.procedure_datetime,
  source.procedure_type_concept_id,
  source.modifier_concept_id,
  source.quantity,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.procedure_source_value,
  source.procedure_source_concept_id,
  source.modifier_source_value
);
